# Section II-A (A.1 + A.2) Evidence-Locked Writer

This Colab notebook drafts **Section II-A** using the **Groq LLM**, strictly grounded in the evidence layer.

## Inputs
- `analysis/II_evidence_v2/section2A_evidence.csv` (plane + model evidence)
- `analysis/II_metric_governance.md` (metric contract)
- `reference_compendium/section_02_fundamentals_template.md` (structure)
- `drafts/section_02_fundamentals_draft.md` (baseline wording)
- `analysis/II_evidence_v2/patch_notes_for_writing.md` (optional anchor reminders)

## Output
- Draft + Consistency Report saved to `analysis/II_evidence_v2/section2A_draft_from_groq.md`.

## Notes
- Only **strong plane-evidence** is used by default (OSNR optical, ESNR electrical).
- Generic SNR without plane cues is excluded.


In [1]:
# Install Groq SDK
!pip install -q groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 2.5 MB/s eta 0:00:00


In [2]:
# Mount Google Drive and set base dir
from google.colab import drive
import os

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST'
os.chdir(BASE_DIR)
print('Working dir:', os.getcwd())


Mounted at /content/drive
Working dir: /content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST


In [3]:
# Load evidence and source files
import pandas as pd
from pathlib import Path

evidence_path = Path('analysis/II_evidence_v2/section2A_evidence.csv')
gov_path = Path('analysis/II_metric_governance.md')
template_path = Path('reference_compendium/section_02_fundamentals_template.md')
draft_path = Path('drafts/section_02_fundamentals_draft.md')
patch_path = Path('analysis/II_evidence_v2/patch_notes_for_writing.md')

df = pd.read_csv(evidence_path)
gov_text = gov_path.read_text(encoding='utf-8', errors='ignore')
template_text = template_path.read_text(encoding='utf-8', errors='ignore')
draft_text = draft_path.read_text(encoding='utf-8', errors='ignore')
patch_text = patch_path.read_text(encoding='utf-8', errors='ignore') if patch_path.exists() else ''

print('Evidence rows:', len(df))


Evidence rows: 824


In [4]:
# Filter evidence for Section II-A
def format_anchor(row):
    heading = row.get('heading_path', '') or 'unknown_heading'
    return "\u27e6{} | {} | L{}\u2013L{}\u27e7 \"{}\"".format(row.paper_id, heading, int(row.line_start), int(row.line_end), row.quote)

# Strong OSNR (optical)
osnr = df[(df.metric == 'OSNR') & (df.plane == 'OPTICAL_PLANE') & (df.strength == 'strong')]
# Strong ESNR (electrical)
esnr = df[(df.metric == 'ESNR') & (df.plane == 'ELECTRICAL_PLANE') & (df.strength == 'strong')]
# Observation models (if present)
coherent = df[df.metric == 'COHERENT_MODEL']
imdd = df[df.metric == 'IMDD_MODEL']

print('OSNR strong:', len(osnr))
print('ESNR strong:', len(esnr))
print('COHERENT_MODEL:', len(coherent))
print('IMDD_MODEL:', len(imdd))

osnr_block = '\n'.join(osnr.apply(format_anchor, axis=1).tolist()[:10])
esnr_block = '\n'.join(esnr.apply(format_anchor, axis=1).tolist()[:10])
coh_block = '\n'.join(coherent.apply(format_anchor, axis=1).tolist()[:10])
imdd_block = '\n'.join(imdd.apply(format_anchor, axis=1).tolist()[:10])

evidence_pack = (
    '[OSNR Evidence]\n' + osnr_block + '\n\n' +
    '[ESNR Evidence]\n' + esnr_block + '\n\n' +
    '[Coherent Model Evidence]\n' + coh_block + '\n\n' +
    '[IM/DD Model Evidence]\n' + imdd_block
)

print('Evidence pack preview:')
print(evidence_pack[:1200])


OSNR strong: 21
ESNR strong: 8
COHERENT_MODEL: 289
IMDD_MODEL: 356
Evidence pack preview:
[OSNR Evidence]
⟦O_ISAC_056 | # Optical ISAC: Fundamental Performance Limits and Transceiver Design | L7–L11⟧ "high optical signal-to-noise ratio (O-SNR)"
⟦O_ISAC_029 | # THz Integrated Sensing and Communication With Full-Photonic Direct LFM Reception and De-Chirping for D-Band Fiber-Wireless Network > ## <span id="page-0-1"></span>I. INTRODUCTION | L45–L49⟧ "optical signal-to-noise ratio (SNR)"
⟦O_ISAC_080 | # Integrated Communication and In-band Spectrum Polarization-Based Sensing via Fraction-Division Non-Orthogonal Multiple Access | L5–L9⟧ "optical signal to noise ratio (OSNR)"
⟦O_ISAC_080 | # V. CONCLUSION | L189–L193⟧ "OSNR improvement can achieve more than 2.3 dB"
⟦O_ISAC_066 | # IV. FORMULA FOR COLLISION DISTORTION ON THE QUANTITIES > ### V. PRE-DISTORTION SCHEME AND DETECTION FOR THE IT SIGNAL. > ### VI. NUMERICAL TESTS | L282–L286⟧ "input optical signal to noise ratio"
⟦O_ISAC_085 | # II

In [6]:
# Generate Section II-A draft with Groq
from groq import Groq
import os
from pathlib import Path
from google.colab import userdata

# Attempt to retrieve API key from Colab secrets or env vars
try:
    api_key = userdata.get('GROQ_API_KEY')
except Exception:
    api_key = os.environ.get('GROQ_API_KEY')

if not api_key:
    raise ValueError("GROQ_API_KEY not found. Please add 'GROQ_API_KEY' to your Colab Secrets (key icon on the left).")

client = Groq(api_key=api_key)

system_prompt = (
    'You are an Evidence-Locked Technical Writer + Consistency Auditor for an IEEE COMST-style survey.\n'
    'Write Section II-A (A.1 and A.2) only.\n'
    'All paper-specific claims MUST be anchored with \u27e6paper_id | heading_path | Lx\u2013Ly\u27e7.\n'
    'No OSNR->ESNR conversion unless explicitly provided.\n'
    'Use academic tone and the required headings.\n'
    'After the draft, output a Consistency Report listing anchors used and any weakened statements.\n'
)

user_prompt = f'''\nDRAFT BASELINE (for tone):\n{draft_text}\n\nTEMPLATE (structure):\n{template_text}\n\nMETRIC GOVERNANCE (contract rules):\n{gov_text}\n\nEVIDENCE PACK:\n{evidence_pack}\n\nPATCH NOTES (optional):\n{patch_text}\n\nOUTPUT FORMAT:\n1) Section II-A text with headings:\n## II. TECHNICAL FUNDAMENTALS OF O-ISAC\n### A. Unified O-ISAC System Model and Integration Paradigms\n#### A.1 Canonical Observation Models and Measurement Planes\n#### A.2 Integration Paradigms (Coexistence -> Cooperation -> Co-design)\n\n2) ### Consistency Report\n- Evidence anchors used: [list]\n- Draft-to-evidence changes: [bullet list]\n- Contract compliance checks: OSNR vs ESNR separation / no conversion / evidence-backed claims\n'''

completion = client.chat.completions.create(
    messages=[
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt}
    ],
    model='llama-3.3-70b-versatile',
    temperature=0.2
)

output = completion.choices[0].message.content
print(output)

out_path = Path('analysis/II_evidence_v2/section2A_draft_from_groq.md')
# Ensure directory exists
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(output, encoding='utf-8')
print('Saved:', out_path)

## II. TECHNICAL FUNDAMENTALS OF O-ISAC

### A.1 Canonical Joint Waveform/Resource Model
We describe a generic joint design space that spans waveform parameters (bandwidth, chirp rate, pilots, coding), optical front-end choices (source, modulation, detection), and sensing task parameters (range/angle/velocity vs fiber spatial granularity). This abstraction allows a single comparison plane across modalities even when implementations differ.

**Generic baseband observation (complex coherent model):**
\[
\mathbf{y}(t)=\mathbf{H}(t;\boldsymbol{\theta})\mathbf{s}(t)+\mathbf{w}(t),
\]
where \(\boldsymbol{\theta}\) collects sensing parameters (delay/range, Doppler, AoA/AoD, vibration state, etc.).

**IM/DD observation (real nonnegative intensity constraint):**
\[
y(t)=\mathcal{R}\,\big(x(t)\ast h(t)\big)+n(t),\qquad x(t)\ge 0,
\]
where \(\mathcal{R}\) is responsivity and \(h(t)\) is the intensity channel impulse response.

Evidence anchors: ⟦O_ISAC_028 | L11-L11⟧, ⟦O_ISAC_029 | L47-L47⟧, ⟦O_I